In [ ]:
"""
LSTM training and evaluation pipeline

This script:
  1. Loads pre-split train / validation / test CSVs from ../dataset (project root)
  2. Uses feature columns from CSVs
  3. Builds sliding-window sequences for the LSTM
  4. Tunes hyperparameters with time-series CV + Optuna (maximize mean validation ROC-AUC)
  5. Trains a final model with early stopping on validation loss (best checkpoint = lowest val loss)
  6. Reports classification metrics on validation and test
  7. Saves ROC curve, confusion matrix, and short-window test prob-vs-time plot under lstm_model/outputs/
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Dict, Tuple

# Project root on path so `lstm_model` imports work when running this file directly
def _project_root() -> Path:
    try:
        return Path(__file__).resolve().parent.parent
    except NameError:
        cwd = Path.cwd().resolve()
        for base in (cwd, cwd.parent):
            if (base / "lstm_model" / "config.py").is_file():
                return base
        raise RuntimeError(
            "Set the Jupyter kernel working directory to the project root "
            "(folder that contains lstm_model/)."
        )


_PROJECT_ROOT = _project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import pandas as pd
import optuna
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, Subset

from lstm_model.data import (
    SequenceDataset,
    load_xy_pair,
    make_sequences,
)
from lstm_model.modules import LSTMClassifier

from lstm_model import config


def plot_roc_curve(
    split_name: str,
    y_true: np.ndarray,
    probs: np.ndarray,
    out_path: Path,
    title_suffix: str = "",
) -> None:
    """Save an ROC curve for a given split."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # ROC/AUC are undefined if only one class is present.
    if len(np.unique(y_true)) < 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.text(
            0.5,
            0.5,
            "ROC curve n/a\n(single class in y_true)",
            ha="center",
            va="center",
            fontsize=12,
        )
        ax.axis("off")
        fig.tight_layout()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved ROC curve to: {out_path}")
        return

    fpr, tpr, _ = roc_curve(y_true, probs)
    auc = roc_auc_score(y_true, probs)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC (AUC={auc:.4f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.set_title(f"ROC Curve - {split_name}{title_suffix}".strip())
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")
    fig.tight_layout()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved ROC curve to: {out_path}")


def plot_confusion_matrix(
    split_name: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    out_path: Path,
) -> None:
    """Save a 2x2 confusion matrix for binary classification."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    classes = ["0", "1"]
    ax.set(
        xticks=np.arange(2),
        yticks=np.arange(2),
        xticklabels=classes,
        yticklabels=classes,
        xlabel="Predicted label",
        ylabel="True label",
        title=f"Confusion Matrix - {split_name}",
    )

    # Annotate cells with counts
    cm_max = float(np.max(cm)) if cm.size else 0.0
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] > cm_max / 2 else "black"
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", color=color, fontsize=12)

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved confusion matrix to: {out_path}")


def plot_test_prob_short_window(
    dates: pd.Series,
    y_true: np.ndarray,
    probs: np.ndarray,
    classification_threshold: float,
    max_points: int,
    out_path: Path,
) -> None:
    # Last ``max_points`` test days: predicted P(up) vs date, threshold line, markers on actual up days
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates

    n = len(probs)
    if n == 0:
        return
    start = max(0, n - int(max_points))
    d = pd.to_datetime(dates.iloc[start:].reset_index(drop=True))
    y = y_true[start:].astype(np.float64).ravel()
    p = probs[start:].astype(np.float64).ravel()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(d, p, color="C0", linewidth=1.5, label="P(up)")
    ax.axhline(classification_threshold, color="gray", linestyle="--", linewidth=1, label=f"threshold={classification_threshold:g}")
    up_mask = y >= 0.5
    if np.any(up_mask):
        ax.scatter(
            d[up_mask],
            np.clip(p[up_mask], 0.0, 1.0),
            color="green",
            s=36,
            zorder=5,
            marker="^",
            label="actual up (y=1)",
        )
    if np.any(~up_mask):
        ax.scatter(
            d[~up_mask],
            np.clip(p[~up_mask], 0.0, 1.0),
            color="C3",
            s=22,
            zorder=4,
            marker="v",
            alpha=0.7,
            label="actual down (y=0)",
        )

    ax.set_ylim(-0.05, 1.08)
    ax.set_xlabel("Date")
    ax.set_ylabel("Predicted probability")
    ax.set_title(f"Test — last {len(d)} days")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    fig.autofmt_xdate(rotation=30, ha="right")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved short-window prob plot to: {out_path}")


# reproducibility
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


# metrics (probabilities from logits via sigmoid)
def compute_metrics(y_true: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    # binary classification metrics for a fixed decision threshold
    # - `probs` are sigmoid probabilities (not logits)
    y_pred = (probs >= threshold).astype(np.int64)
    out: Dict[str, float] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    # AUC requires both classes present in y_true
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, probs))
    else:
        out["roc_auc"] = float("nan")
    return out


def print_metrics_block(name: str, metrics: Dict[str, float]) -> None:
    print(f"\n=== {name} ===")
    for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        if k not in metrics:
            continue
        print(f"  {k:16s}: {metrics[k]:.4f}" if not np.isnan(metrics[k]) else f"  {k:16s}: n/a")


# training / evaluation steps
@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
) -> Tuple[float, np.ndarray, np.ndarray]:
    # returns mean loss, true labels, predicted probabilities (CPU tensors only)
    model.eval()
    total_loss = 0.0
    n = 0
    ys: list[np.ndarray] = []
    ps: list[np.ndarray] = []

    for xb, yb in loader:
        logits = model(xb)
        loss = loss_fn(logits, yb)
        total_loss += loss.item() * len(yb)
        n += len(yb)
        prob = torch.sigmoid(logits)
        ys.append(yb.detach().numpy())
        ps.append(prob.detach().numpy())

    y_true = np.concatenate(ys, axis=0)
    probs = np.concatenate(ps, axis=0)
    return total_loss / max(n, 1), y_true, probs


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    max_grad_norm: float = 1.0,
) -> float:
    model.train()
    total = 0.0
    n = 0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        # Gradient clipping improves stability on recurrent models
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / max(n, 1)


def train_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    pos_weight: torch.Tensor,
    lr: float,
    weight_decay: float,
    max_epochs: int,
    patience: int,
    max_grad_norm: float = 1.0,
    lr_plateau_factor: float = 0.5,
    lr_plateau_patience: int = 3,
    log_every_n_epochs: int = 5,
) -> Tuple[nn.Module, Dict[str, list[float]]]:
    # Train with binary cross-entropy with logits (numerically stable).
    # pos_weight handles class imbalance (weighted positive class).
    #
    # Optimizer: Adam. ReduceLROnPlateau (lr_plateau_*) reduces LR when validation loss plateaus.
    #
    # Early stopping and the returned weights use the lowest validation loss so far
    # (patience epochs without improvement).
    #
    # Returns the model (best validation-loss weights) and history dict with
    # train_losses / val_losses per epoch.
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_plateau_factor,
        patience=lr_plateau_patience,
    )

    best_state = None
    best_val_loss = float("inf")
    bad_epochs = 0
    train_losses: list[float] = []
    val_losses: list[float] = []

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, loss_fn, max_grad_norm=max_grad_norm
        )
        val_loss, y_val, p_val = evaluate_model(model, val_loader, loss_fn)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_metrics = compute_metrics(y_val, p_val)
        val_auc = val_metrics.get("roc_auc", float("nan"))

        if np.isfinite(val_loss):
            scheduler.step(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
        else:
            bad_epochs += 1

        if (epoch + 1) % log_every_n_epochs == 0 or epoch == 0:
            cur_lr = optimizer.param_groups[0]["lr"]
            print(
                f"  epoch {epoch+1:03d}/{max_epochs}  lr={cur_lr:.2e}  "
                f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
                f"val_auc={val_auc:.4f}"
            )

        if bad_epochs >= patience:
            print(f"  early stopping at epoch {epoch+1} (no val loss improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    history = {"train_losses": train_losses, "val_losses": val_losses}
    return model, history


def run_cv_score(
    X_train: np.ndarray,
    y_train: np.ndarray,
    hparams: Dict[str, Any],
    n_splits: int,
    inner_epochs: int,
    pos_weight_scalar: float,
) -> float:
    # TimeSeriesSplit cross-validation on training sequences only.
    # Returns mean validation ROC-AUC across folds.

    seq_len = int(hparams["seq_len"])
    hidden_dim = int(hparams["hidden_dim"])
    num_layers = int(hparams["num_layers"])
    dropout = float(hparams["dropout"])
    lr = float(hparams["lr"])
    weight_decay = float(hparams["weight_decay"])
    batch_size = int(hparams["batch_size"])

    X_seq, y_seq = make_sequences(X_train, y_train, seq_len)
    n = len(y_seq)

    # don't run cv if there is not enough sequences
    if n < n_splits + 1:
        return float("-inf")

    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_val_aucs: list[float] = []
    input_dim = X_train.shape[1]
    pos_w = torch.tensor([pos_weight_scalar])

    for fold, (train_idx, val_idx) in enumerate(tscv.split(np.arange(n))):
        ds = SequenceDataset(X_seq, y_seq)
        tr_loader = DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True, drop_last=False)
        va_loader = DataLoader(Subset(ds, val_idx), batch_size=batch_size, shuffle=False, drop_last=False)

        model = LSTMClassifier(input_dim, hidden_dim, num_layers, dropout)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        for _ in range(inner_epochs):
            train_one_epoch(model, tr_loader, optimizer, loss_fn)

        _, y_val_fold, p_val_fold = evaluate_model(model, va_loader, loss_fn)
        # ROC-AUC is undefined if only one class is present.
        if len(np.unique(y_val_fold)) > 1:
            fold_val_aucs.append(float(roc_auc_score(y_val_fold, p_val_fold)))

    if not fold_val_aucs:
        return float("-inf")

    return float(np.mean(fold_val_aucs))


# optuna objective (optional)
def _suggest_optuna_hparams(trial: Any, space: Dict[str, tuple]) -> Dict[str, Any]:
    # sample hyperparameters for one trial using config.optuna_search_space layout
    out: Dict[str, Any] = {}
    for name, spec in space.items():
        kind = spec[0]
        if kind == "int":
            out[name] = trial.suggest_int(name, spec[1], spec[2])
        elif kind == "float":
            lo, hi, log = spec[1], spec[2], spec[3]
            out[name] = trial.suggest_float(name, lo, hi, log=log)
        elif kind == "categorical":
            out[name] = trial.suggest_categorical(name, list(spec[1]))
        else:
            raise ValueError(f"unknown optuna space kind {kind!r} for {name!r}")
    return out

# function to tune hyperparameters with optuna
def optuna_tune(
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_trials: int,
    n_splits: int,
    inner_epochs: int,
    seed: int,
) -> Dict[str, Any]:

    # Class imbalance: weight for positive class in BCE
    pos = float(np.sum(y_train == 1.0))
    neg = float(np.sum(y_train == 0.0))
    pos_weight_scalar = (neg / max(pos, 1.0))

    def objective(trial: optuna.Trial) -> float:
        set_seed(seed + trial.number)
        hp = _suggest_optuna_hparams(trial, config.optuna_search_space)
        
        return run_cv_score(
            X_train,
            y_train,
            hp,
            n_splits=n_splits,
            inner_epochs=inner_epochs,
            pos_weight_scalar=pos_weight_scalar,
        )

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=config.optuna_show_progress_bar)

    best = study.best_params
    print("\n[Optuna] Best trial:")
    print(f"  value (mean val ROC-AUC): {study.best_value:.6f}")
    print(f"  params: {best}")
    return dict(best)


def main() -> int:
    set_seed(config.seed)

    # load data 
    data_dir = Path(config.dataset_dir)
    paths = {
        "train": (data_dir / config.train_features_csv, data_dir / config.train_targets_csv),
        "val": (data_dir / config.val_features_csv, data_dir / config.val_targets_csv),
        "test": (data_dir / config.test_features_csv, data_dir / config.test_targets_csv),
    }

    X_train_df, y_train, _ = load_xy_pair(*paths["train"])
    X_val_df, y_val, _ = load_xy_pair(*paths["val"])
    X_test_df, y_test, dates_test = load_xy_pair(*paths["test"])

    # print data sizes for checking
    print(f"Train rows: {len(X_train_df)}  features: {X_train_df.shape[1]}")
    print(f"Val rows:   {len(X_val_df)}")
    print(f"Test rows:  {len(X_test_df)}")

    # get feature columns
    feature_cols = list(X_train_df.columns)

    # convert to numpy arrays
    X_train = X_train_df.to_numpy(dtype=np.float32)
    X_val = X_val_df.to_numpy(dtype=np.float32)
    X_test = X_test_df.to_numpy(dtype=np.float32)

    # class imbalance used for loss
    pos = float(np.sum(y_train == 1.0))
    neg = float(np.sum(y_train == 0.0))
    # make sure no divide by 0
    pos_weight_scalar = neg / max(pos, 1.0)
    pos_weight = torch.tensor([pos_weight_scalar])
    # for checking
    print(f"Train class balance: pos={int(pos)} neg={int(neg)}  BCE pos_weight={pos_weight_scalar:.3f}")

    # hyperparameter search (time-series CV) or defaults
    hparams = dict(config.default_model_hparams)
    skip_tune = config.skip_hyperparameter_tuning

    if not skip_tune:
        # hyperparameter tuning loop
        hparams.update(
            optuna_tune(
                X_train,
                y_train,
                n_trials=config.n_trials,
                n_splits=config.cv_splits,
                inner_epochs=config.inner_epochs,
                seed=config.seed,
            )
        )
    else:
        # skip hyperparameter tuning
        print("Skipping hyperparameter tuning; using lstm_model.config.default_model_hparams.")

    # print final hyperparameters for checking
    print(f"\nUsing hyperparameters: {hparams}")

    # final sequences: train on all training days; validate/test on their splits

    # get sequence length
    seq_len = int(hparams["seq_len"])

    # make sequences
    X_seq_train, y_seq_train = make_sequences(X_train, y_train, seq_len)
    X_seq_val, y_seq_val = make_sequences(X_val, y_val, seq_len)
    X_seq_test, y_seq_test = make_sequences(X_test, y_test, seq_len)

    # print sequence counts for checking
    print(
        f"Sequence counts - train: {len(y_seq_train)}, val: {len(y_seq_val)}, test: {len(y_seq_test)} "
        f"(seq_len={seq_len})"
    )

    # create data loaders
    loader_kw = {"num_workers": config.num_workers}
    train_loader = DataLoader(
        SequenceDataset(X_seq_train, y_seq_train),
        batch_size=int(hparams["batch_size"]),
        shuffle=True,
        drop_last=False,
        **loader_kw,
    )
    val_loader = DataLoader(
        SequenceDataset(X_seq_val, y_seq_val),
        batch_size=int(hparams["batch_size"]),
        shuffle=False,
        drop_last=False,
        **loader_kw,
    )
    test_loader = DataLoader(
        SequenceDataset(X_seq_test, y_seq_test),
        batch_size=int(hparams["batch_size"]),
        shuffle=False,
        drop_last=False,
        **loader_kw,
    )

    model = LSTMClassifier(
        input_dim=X_train.shape[1],
        hidden_dim=int(hparams["hidden_dim"]),
        num_layers=int(hparams["num_layers"]),
        dropout=float(hparams["dropout"]),
    )

    print("\nTraining final model with early stopping on validation loss")
    print(f"  Optimizer: Adam  lr={float(hparams['lr']):.2e}  grad_clip={config.max_grad_norm}")
    model, history = train_with_early_stopping(
        model,
        train_loader,
        val_loader,
        pos_weight=pos_weight,
        lr=float(hparams["lr"]),
        weight_decay=float(hparams["weight_decay"]),
        max_epochs=config.final_epochs,
        patience=config.early_stopping_patience,
        max_grad_norm=config.max_grad_norm,
        lr_plateau_factor=config.lr_plateau_factor,
        lr_plateau_patience=config.lr_plateau_patience,
        log_every_n_epochs=config.log_every_n_epochs,
    )

    loss_fn_eval = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    _, y_va, p_va = evaluate_model(model, val_loader, loss_fn_eval)
    _, y_te, p_te = evaluate_model(model, test_loader, loss_fn_eval)

    thr = float(config.classification_threshold)

    m_va = compute_metrics(y_va, p_va, threshold=thr)
    m_te = compute_metrics(y_te, p_te, threshold=thr)

    print_metrics_block("Validation", m_va)
    print_metrics_block("Test", m_te)

    export_path = Path(config.export_test_preds_path) if config.export_test_preds_path else None
    if export_path is not None:
        # Targets align with rows seq_len-1 .. end of the aligned test frame
        dates_aligned = dates_test.iloc[seq_len - 1 :].reset_index(drop=True)
        pred_df = pd.DataFrame(
            {
                "Date": dates_aligned.values,
                "prob_up": p_te.astype(np.float64),
                "y_true": y_te.astype(np.float64),
                "y_pred": (p_te >= thr).astype(np.int64),
            }
        )
        export_path.parent.mkdir(parents=True, exist_ok=True)
        pred_df.to_csv(export_path, index=False)
        print(f"\nWrote test predictions to: {export_path}")

    # Save checkpoints and plots under lstm_model/outputs/
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dates_te_aligned = dates_test.iloc[seq_len - 1 :].reset_index(drop=True)

    try:
        # Test plots
        plot_roc_curve(
            "Test",
            y_te,
            p_te,
            out_dir / "roc_curve_test.png",
        )
        plot_confusion_matrix(
            "Test",
            y_te,
            (p_te >= thr).astype(np.int64),
            out_dir / "confusion_matrix_test.png",
        )
        plot_test_prob_short_window(
            dates_te_aligned,
            y_te,
            p_te,
            thr,
            int(getattr(config, "prob_short_window_max_points", 80)),
            out_dir / str(getattr(config, "prob_short_window_filename", "test_prob_short_window.png")),
        )
    except ImportError:
        print("[warn] matplotlib not installed; skipping plots. pip install matplotlib")

    torch.save(
        {
            "model_state": model.state_dict(),
            "hparams": hparams,
            "feature_cols": feature_cols,
            "seq_len": seq_len,
            "train_losses": history["train_losses"],
            "val_losses": history["val_losses"],
        },
        out_dir / config.checkpoint_filename,
    )
    print(f"\nSaved checkpoint to: {out_dir / config.checkpoint_filename}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
